# Train Downstream Fusion Models

Run this notebook after the CBM notebook has produced full-size `concept_vectors.csv` in the Drive `CLARIFY` folder.

Inputs:
- `data/lyrics/master_lyrics_features.csv`
- `CLARIFY/data/lyrics/concept_vectors.csv`
- `data/audio/master_audio_features.csv`

Outputs saved to Google Drive under `/content/drive/MyDrive/CLARIFY`:
- `CLARIFY/data/lyrics/downstream_fusion_table.csv`
- `CLARIFY/lyrics/model_outputs/downstream_hit_score_models.joblib` if a hit-score or rank target exists
- `CLARIFY/lyrics/model_outputs/downstream_recommender_index.joblib`
- `CLARIFY/lyrics/model_outputs/downstream_fusion_metadata.json`

What it does:
- mounts Google Drive
- joins lyric, concept, and Librosa audio streams by normalized song title and artist occurrence
- compares familiar downstream model variants in one place
- builds the final multi-stream recommender index
- does not train the CBM or call Spotify APIs

## 1. Setup

Imports sklearn, pandas, numpy, and joblib for downstream tabular modeling.

In [ ]:
!pip install -q pandas numpy scikit-learn joblib

In [ ]:
from pathlib import Path
import json
import math
import re

import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as exc:
    print('Drive mount skipped:', exc)

## 2. Paths

Finds input CSVs from the repo or Drive, reads CBM concept vectors from the Drive `CLARIFY` folder, and saves downstream artifacts back to `CLARIFY`.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/CLARIFY')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    DRIVE_ROOT,
    Path('/content/DS3-CLARIFY'),
    Path('/content/drive/MyDrive/DS3-CLARIFY'),
]
PROJECT_ROOT = next(
    (root for root in candidate_roots if (root / 'data' / 'audio' / 'master_audio_features.csv').exists()),
    DRIVE_ROOT,
)

LYRIC_DATA_DIR = PROJECT_ROOT / 'data' / 'lyrics'
AUDIO_DATA_DIR = PROJECT_ROOT / 'data' / 'audio'
DRIVE_LYRIC_DATA_DIR = DRIVE_ROOT / 'data' / 'lyrics'
MODEL_DIR = DRIVE_ROOT / 'lyrics' / 'model_outputs'
DRIVE_LYRIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

LYRIC_FEATURE_PATH = LYRIC_DATA_DIR / 'master_lyrics_features.csv'
if not LYRIC_FEATURE_PATH.exists():
    LYRIC_FEATURE_PATH = LYRIC_DATA_DIR / 'final_CBM_input_data.csv'
CONCEPT_VECTORS_PATH = DRIVE_LYRIC_DATA_DIR / 'concept_vectors.csv'
if not CONCEPT_VECTORS_PATH.exists():
    CONCEPT_VECTORS_PATH = LYRIC_DATA_DIR / 'concept_vectors.csv'
AUDIO_FEATURE_PATH = AUDIO_DATA_DIR / 'master_audio_features.csv'

TARGET_COLUMN = 'HitScore'
RANK_COLUMNS = ['Billboard_Rank', 'Rank', 'Peak_Rank', 'peak_rank']

FUSION_TABLE_PATH = DRIVE_LYRIC_DATA_DIR / 'downstream_fusion_table.csv'
HIT_SCORE_MODELS_PATH = MODEL_DIR / 'downstream_hit_score_models.joblib'
RECOMMENDER_INDEX_PATH = MODEL_DIR / 'downstream_recommender_index.joblib'
METADATA_PATH = MODEL_DIR / 'downstream_fusion_metadata.json'

print('Project root:', PROJECT_ROOT)
print('Drive artifact root:', DRIVE_ROOT)
for label, path in {'lyrics': LYRIC_FEATURE_PATH, 'concept_vectors': CONCEPT_VECTORS_PATH, 'audio': AUDIO_FEATURE_PATH}.items():
    print(label, path, path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing {label}: {path}')
print('Fusion table saves to:', FUSION_TABLE_PATH)
print('Downstream model artifacts save to:', MODEL_DIR)

## 3. Columns and Helpers

Defines the lyric, concept, and Librosa audio feature columns plus the title/artist join key.

In [ ]:
LYRIC_IDENTITY_COLUMNS = ['SONG_TITLE', 'ARTIST_NAME', 'SONG_ID']
HANDCRAFTED_FEATURE_COLUMNS = [
    'Word_Count', 'Unique_Word_Count', 'Repetition_Score', 'Average_Line_Length',
    'Vocabulary_Diversity', 'Title_Repetition', 'Explicit_Word_Count', 'Sentiment_Score',
    'Positive_Score', 'Negative_Score', 'Emotional_Intensity',
]
LIBROSA_FEATURE_COLUMNS = (
    ['tempo']
    + [f'mfcc_{i}' for i in range(1, 14)]
    + [f'chroma_mean_{i}' for i in range(1, 13)]
    + [f'chroma_std_{i}' for i in range(1, 13)]
    + ['spectral_centroid']
)
OPTIONAL_AUDIO_METADATA_FEATURES = ['year']

def normalize_text(value):
    return re.sub(r'[^a-z0-9]+', '', str(value).lower())

def make_song_artist_key(df, title_col, artist_col):
    return df[title_col].map(normalize_text) + '|' + df[artist_col].map(normalize_text)

def embedding_sort_key(column):
    match = re.search(r'(\d+)$', column)
    return int(match.group(1)) if match else -1

def get_embedding_columns(df):
    return sorted([col for col in df.columns if re.fullmatch(r'BERT_Embedding_\d+', col)], key=embedding_sort_key)

def require_columns(df, required, name):
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f'{name} missing columns: {missing}')

def ensure_song_id(df, prefix):
    df = df.copy()
    if 'SONG_ID' not in df.columns:
        df['SONG_ID'] = [f'{prefix}_{idx:05d}' for idx in range(len(df))]
    return df

def add_join_occurrence(df):
    df = df.copy()
    df['_join_occurrence'] = df.groupby('_join_key').cumcount()
    return df

## 4. Load Component Outputs

Loads the lyric table, CBM concept vectors, and master Librosa audio table.

In [ ]:
lyrics_df = ensure_song_id(pd.read_csv(LYRIC_FEATURE_PATH), 'lyrics_full')
concept_vectors = ensure_song_id(pd.read_csv(CONCEPT_VECTORS_PATH), 'lyrics_full')
audio_df = ensure_song_id(pd.read_csv(AUDIO_FEATURE_PATH), 'audio')

embedding_columns = get_embedding_columns(lyrics_df)
LYRIC_INPUT_COLUMNS = HANDCRAFTED_FEATURE_COLUMNS + embedding_columns
CONCEPT_VECTOR_COLUMNS = [col for col in concept_vectors.columns if col.startswith('Concept_')]

require_columns(lyrics_df, LYRIC_IDENTITY_COLUMNS + LYRIC_INPUT_COLUMNS, 'lyrics feature table')
require_columns(concept_vectors, LYRIC_IDENTITY_COLUMNS + CONCEPT_VECTOR_COLUMNS, 'concept vector table')
require_columns(audio_df, ['SONG_ID', 'SONG_TITLE', 'ARTIST_NAME'] + LIBROSA_FEATURE_COLUMNS, 'master audio feature table')

lyrics_df['_join_key'] = make_song_artist_key(lyrics_df, 'SONG_TITLE', 'ARTIST_NAME')
concept_vectors['_join_key'] = make_song_artist_key(concept_vectors, 'SONG_TITLE', 'ARTIST_NAME')
audio_df['_join_key'] = make_song_artist_key(audio_df, 'SONG_TITLE', 'ARTIST_NAME')
lyrics_df = add_join_occurrence(lyrics_df)
concept_vectors = add_join_occurrence(concept_vectors)
audio_df = add_join_occurrence(audio_df)

print('Lyrics rows:', len(lyrics_df))
print('Concept-vector rows:', len(concept_vectors))
print('Audio rows:', len(audio_df))
print('Lyric feature columns:', len(LYRIC_INPUT_COLUMNS))
print('Concept vector columns:', len(CONCEPT_VECTOR_COLUMNS))

## 5. Build Fusion Table

Creates the shared modeling table with separate lyric IDs and audio IDs, then saves it for inspection.

In [ ]:
def prepare_librosa_audio_features(audio_df):
    audio = audio_df.copy().rename(columns={'SONG_ID': 'AUDIO_SONG_ID'})
    feature_columns = [col for col in OPTIONAL_AUDIO_METADATA_FEATURES + LIBROSA_FEATURE_COLUMNS if col in audio.columns]
    for col in feature_columns:
        audio[col] = pd.to_numeric(audio[col], errors='coerce')
    out = audio[['AUDIO_SONG_ID', 'SONG_TITLE', 'ARTIST_NAME', '_join_key', '_join_occurrence'] + feature_columns].copy()
    out = out.rename(columns={col: f'Audio_{col}' for col in feature_columns})
    audio_input_columns = [f'Audio_{col}' for col in feature_columns]
    out[audio_input_columns] = out[audio_input_columns].fillna(out[audio_input_columns].median(numeric_only=True))
    return out, audio_input_columns

audio_features, AUDIO_INPUT_COLUMNS = prepare_librosa_audio_features(audio_df)
lyrics_features = lyrics_df[LYRIC_IDENTITY_COLUMNS + ['_join_key', '_join_occurrence'] + LYRIC_INPUT_COLUMNS].rename(columns={'SONG_ID': 'LYRIC_SONG_ID'})
concept_features = concept_vectors[LYRIC_IDENTITY_COLUMNS + ['_join_key', '_join_occurrence'] + CONCEPT_VECTOR_COLUMNS].rename(columns={'SONG_ID': 'LYRIC_SONG_ID'})

fusion = lyrics_features.merge(
    concept_features[['LYRIC_SONG_ID', '_join_key', '_join_occurrence'] + CONCEPT_VECTOR_COLUMNS],
    on=['LYRIC_SONG_ID', '_join_key', '_join_occurrence'],
    how='inner',
    validate='one_to_one',
)
fusion = fusion.merge(
    audio_features,
    on=['_join_key', '_join_occurrence'],
    how='inner',
    suffixes=('', '_audio'),
    validate='one_to_one',
)
fusion = fusion.drop(columns=[col for col in ['SONG_TITLE_audio', 'ARTIST_NAME_audio'] if col in fusion.columns])

audio_match_keys = set(zip(audio_df['_join_key'], audio_df['_join_occurrence']))
lyric_keys = list(zip(lyrics_df['_join_key'], lyrics_df['_join_occurrence']))
missing_audio = lyrics_df.loc[[key not in audio_match_keys for key in lyric_keys], LYRIC_IDENTITY_COLUMNS]
if len(missing_audio):
    print(f'Lyric rows without matching Librosa audio: {len(missing_audio)}')
    display(missing_audio.head(10))

fusion.to_csv(FUSION_TABLE_PATH, index=False)
print('Fusion rows:', len(fusion))
print('Audio feature columns:', len(AUDIO_INPUT_COLUMNS))
print('Saved fusion table:', FUSION_TABLE_PATH)
fusion[['SONG_TITLE', 'ARTIST_NAME', 'LYRIC_SONG_ID', 'AUDIO_SONG_ID'] + AUDIO_INPUT_COLUMNS[:5]].head()

## 6. Optional Hit-Score Models

If a target exists, compares `concepts_only`, `lyrics_only`, `audio_only_librosa`, `audio_lyrics`, and `audio_lyrics_concepts`.

In [ ]:
def attach_downstream_target(fusion, lyrics_df, audio_df):
    if TARGET_COLUMN in lyrics_df.columns:
        target = lyrics_df[['_join_key', TARGET_COLUMN]].copy()
        target[TARGET_COLUMN] = pd.to_numeric(target[TARGET_COLUMN], errors='coerce')
        return fusion.merge(target, on='_join_key', how='left', validate='one_to_one'), TARGET_COLUMN
    if TARGET_COLUMN in audio_df.columns:
        target = audio_df[['_join_key', TARGET_COLUMN]].copy()
        target[TARGET_COLUMN] = pd.to_numeric(target[TARGET_COLUMN], errors='coerce')
        return fusion.merge(target, on='_join_key', how='left', validate='one_to_one'), TARGET_COLUMN
    for rank_col in RANK_COLUMNS:
        source = lyrics_df if rank_col in lyrics_df.columns else audio_df if rank_col in audio_df.columns else None
        if source is not None:
            target = source[['_join_key', rank_col]].copy()
            rank = pd.to_numeric(target[rank_col], errors='coerce')
            target[TARGET_COLUMN] = (101 - rank).clip(lower=0, upper=100) / 100.0
            return fusion.merge(target[['_join_key', TARGET_COLUMN]], on='_join_key', how='left', validate='one_to_one'), TARGET_COLUMN
    return fusion.copy(), None

model_ready, target_column = attach_downstream_target(fusion, lyrics_df, audio_df)
hit_score_models = {}
hit_score_metrics = None

if target_column is None:
    print('No true downstream target found. Skipping hit-score training.')
else:
    model_df = model_ready.dropna(subset=[target_column]).reset_index(drop=True)
    if len(model_df) < 20:
        print(f'Only {len(model_df)} target rows. Skipping until more target labels exist.')
    else:
        feature_sets = {
            'concepts_only': CONCEPT_VECTOR_COLUMNS,
            'lyrics_only': LYRIC_INPUT_COLUMNS,
            'audio_only_librosa': AUDIO_INPUT_COLUMNS,
            'audio_lyrics': AUDIO_INPUT_COLUMNS + LYRIC_INPUT_COLUMNS,
            'audio_lyrics_concepts': AUDIO_INPUT_COLUMNS + LYRIC_INPUT_COLUMNS + CONCEPT_VECTOR_COLUMNS,
        }
        rows = []
        for name, columns in feature_sets.items():
            X = model_df[columns].astype(float)
            y = model_df[target_column].astype(float)
            model = Pipeline([('scaler', StandardScaler()), ('regressor', Ridge(alpha=10.0))])
            cv = KFold(n_splits=min(5, len(model_df)), shuffle=True, random_state=SEED)
            pred = np.zeros(len(model_df))
            for train_idx, test_idx in cv.split(X):
                model.fit(X.iloc[train_idx], y.iloc[train_idx])
                pred[test_idx] = model.predict(X.iloc[test_idx])
            rows.append({'Model': name, 'Target': target_column, 'Rows': len(model_df), 'Features': len(columns), 'MAE': mean_absolute_error(y, pred), 'RMSE': mean_squared_error(y, pred) ** 0.5, 'R2': r2_score(y, pred)})
            model.fit(X, y)
            hit_score_models[name] = {'model': model, 'feature_columns': columns, 'target_column': target_column}
        hit_score_metrics = pd.DataFrame(rows)
        joblib.dump(hit_score_models, HIT_SCORE_MODELS_PATH)
        display(hit_score_metrics.round(4))

## 7. Recommender Index

Combines weighted audio, lyric, and concept blocks into one cosine similarity recommender.

In [ ]:
SIMILARITY_WEIGHTS = {'audio': 0.35, 'lyrics': 0.35, 'concepts': 0.30}

def weighted_block(df, columns, weight):
    scaler = StandardScaler()
    values = df[columns].astype(float).to_numpy()
    values = np.nan_to_num(values, nan=0.0)
    return math.sqrt(weight) * scaler.fit_transform(values), scaler

audio_block, audio_scaler = weighted_block(fusion, AUDIO_INPUT_COLUMNS, SIMILARITY_WEIGHTS['audio'])
lyric_block, lyric_scaler = weighted_block(fusion, LYRIC_INPUT_COLUMNS, SIMILARITY_WEIGHTS['lyrics'])
concept_block, concept_scaler = weighted_block(fusion, CONCEPT_VECTOR_COLUMNS, SIMILARITY_WEIGHTS['concepts'])
similarity_vectors = np.concatenate([audio_block, lyric_block, concept_block], axis=1)

recommender = NearestNeighbors(metric='cosine', algorithm='brute')
recommender.fit(similarity_vectors)
payload = {
    'model': recommender,
    'vectors': similarity_vectors,
    'song_metadata': fusion[['SONG_TITLE', 'ARTIST_NAME', 'LYRIC_SONG_ID', 'AUDIO_SONG_ID']].reset_index(drop=True),
    'audio_columns': AUDIO_INPUT_COLUMNS,
    'lyric_columns': LYRIC_INPUT_COLUMNS,
    'concept_columns': CONCEPT_VECTOR_COLUMNS,
    'weights': SIMILARITY_WEIGHTS,
    'scalers': {'audio': audio_scaler, 'lyrics': lyric_scaler, 'concepts': concept_scaler},
}
joblib.dump(payload, RECOMMENDER_INDEX_PATH)

def recommend_by_row(row_index, k=5):
    distances, indices = recommender.kneighbors(similarity_vectors[[row_index]], n_neighbors=min(k + 1, len(fusion)))
    rows = []
    for distance, idx in zip(distances[0], indices[0]):
        if idx == row_index:
            continue
        rows.append({'SONG_TITLE': fusion.loc[idx, 'SONG_TITLE'], 'ARTIST_NAME': fusion.loc[idx, 'ARTIST_NAME'], 'Similarity': 1 - distance})
    return pd.DataFrame(rows)

recommend_by_row(0, k=5)

## 8. Save Metadata

Writes run metadata showing row counts, feature counts, artifact paths, and whether a target was available.

In [ ]:
metadata = {
    'rows': {'lyrics': int(len(lyrics_df)), 'audio': int(len(audio_df)), 'concept_vectors': int(len(concept_vectors)), 'fusion': int(len(fusion))},
    'feature_counts': {'audio_librosa': len(AUDIO_INPUT_COLUMNS), 'lyrics': len(LYRIC_INPUT_COLUMNS), 'concepts': len(CONCEPT_VECTOR_COLUMNS)},
    'requires_cbm_first': True,
    'join_strategy': 'normalized_song_title_plus_artist',
    'uses_spotify_api': False,
    'uses_spotify_ids': False,
    'target_column': target_column,
    'artifacts': {
        'fusion_table': str(FUSION_TABLE_PATH),
        'hit_score_models': str(HIT_SCORE_MODELS_PATH) if hit_score_models else None,
        'recommender_index': str(RECOMMENDER_INDEX_PATH),
    },
    'similarity_weights': SIMILARITY_WEIGHTS,
}
METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
metadata